# **รายละเอียดผู้สมัคร**
ผู้สมัครหมายเลข 600659 นายรัชพล สม่าหลี <br>
#นักศึกษาสาขาสถิติ คณะวิทยาศาสตร์และเทคโนโลยี มหาวิทยาลัยธรรมศาสตร์

ต้นฉบับ :

https://colab.research.google.com/drive/1g9vWI9frykAJ0hGe2Le6roLs9sarEIzd?usp=sharing

---

## **วิธีการทดลอง และผลการทดลอง**

ตรวจสอบลักษณะข้อมูลที่ได้รับ
- Knowledges : แบรนด์ในเครือ, แบรนด์พันธมิต, รายละเอียดร้าน ลักษณะของข้อมูล (เป็น .MD)
- จะต้องตอบคำถาม multiple choies 10 ข้อ : ตัวเลือกคำตอบ 1-8, 9 สำหรับไม่พบในฐานข้อมูล และ 10 หากคำถามไม่เกี่ยวข้องกับร้านฟ้าใหม่



**1. ขั้นตอนการ Embedding สำหรับการเข้า Encode ข้อความ**
- โมเดลที่นำมาใช้ทดสอบ (Candidates) โดยกำหนดการทดลองให้มีความแตกต่างคือใช้ โมเดลใหญ่ และ โมเดลเล็ก
    - bge-m3 (ตัวแทนโมเดลเล็ก)
        สาเหตุที่เลือกมา พบว่าได้รับการแนะนำจาก Community และมีผลการ Benchmark ที่ดีสำหรับงาน Retrieve (จากการทดสอบ Thai retrieval benchmark)
        - Source: https://github.com/mrpeerat/Thai-Sentence-Vector-Benchmark
    - Qwen3-Embedding-4B (ตัวแทนโมเดลใหญ่)
        สาเหตุที่เลือกมา พบว่าผลการ Benchmark Text Embedding ภาษาไทยเทียบกับโมเดลหลายๆขนาดหลายๆรุ่น เป็นโมเดลขนาดใหญ่ (4B) ที่ได้ผลลัพธ์การประเมินที่ดีสุด
        - Source: https://github.com/anusoft/thai-mteb-leaderboard
    - เปรียบเทียบคะแนนการทำข้อสอบโดยกำหนดโมเดล KBTG และ Prompt เดียวกันพบว่าโมเดล bge-m3 จะได้คะแนนสอบในช่วง **80-85%** ในขณะที่ โมเดล Qwen3 4b ได้รับคะแนนอยู่ในช่วง (90%-93%) สำหรับ public score ดังนั้นในการสร้างระบบ RAG จะใช้โมเดล Qwen3 4b เป็นโมเดล Embedding
- กลยุทธการ Chunking จะทำการ Tuning (ทดลองเปลี่ยนค่าไปเรื่อยๆ) เพื่อหาค่า:
    - Chunking Size : [256,512,1024,2048] -> พบว่า 1024 ดีที่สุด
    - Overlap : จากการศึกษาของ Nbidia ควรใช้ค่าที่ 10% - 20% อย่างไรก็ตามพบว่าการตั้งค่าที่ 30% โมเดลจะตอบคำถามได้คะแนนสูงสุด
        - Source https://developer.nvidia.com/blog/finding-the-best-chunking-strategy-for-accurate-ai-responses
- การ Retrieveal:
    - ทดลองทั้งวิธี Dense, BM25 และ Hybrid -> Benchmark ด้วยคะแนนสอบ ผลการศึกษาพบว่าวิธี Hybrid มักจะได้คะแนนสอบสูงที่สุด


**2. การ Retrieveal เพื่อหาวิธีที่จะทำให้การ Retrieval มีประสิทธิภาพสูงที่สุด**
- วิธีการ Retrieve
    - ทดลองทั้งวิธี Dense, BM25 และ Hybrid -> Benchmark ด้วยคะแนนสอบ ผลการศึกษาพบว่าวิธี Hybrid แบบกำหนดน้ำหนัก 0.7 ให้กับ Dense และ 0.3 ให้กับ BM 25 มักจะได้คะแนนสอบสูงที่สุด
- วิธีการ Reranking
    - สำหรับขั้นตอนการประเมินผล Embedding ระหว่าง bge-m3 และ Qwen3-4b จะมีการใช้ Reranker ของโมเดลร่วมด้วยคือ
    - โมเดล bge-m3 จะใช้โมเดล bge-reranker-v2-m3 ร่วมด้วย (คะแนน 80%-85%)
    - โมเดล Qwen3-4b จะใช้โมเดล Qwen3-Reranker-4B ร่วมด้วย (คะแนน 90%-93%)
    - สามารถเข้าถึงคะแนนแต่ละรอบได้จาก Kaggle
- **ผลการทดลอง** ใช้โมเดล Qwen3-4b ร่วมกับ Reranker Qwen3-Reranker-4B ในการ Retrieve

**3. โมเดลสำหรับการสร้างข้อความ**
- โมเดลที่นำมาใช้ทดสอบประสิทธิภาพการ Generate และตอบคำถาม ได้แก่:
    - OpenThaiGPT-ThaiLLM-8B-instruct-v7.2,
    - Pathumma-ThaiLLM-qwen3-8b-think-3.0.0
    - Typhoon-S-ThaiLLM-8B-Instruct
    - THaLLE-0.2-ThaiLLM-8b-fa
- **โมเดล LLM สำหรับ Generateที่นำมาใช้จริงโดยใช้เกณฑ์ : ความสามารถในการสร้างข้อความตามคำสั่ง และ คะแนนจากการตอบคำถาม:
    - **THaLLE-0.2-ThaiLLM-8b-fa**


## สรุปผลการทดลอง : วิธีที่ใช้ในการสร้าง RAG System
- Embedding Model : Qwen3-4b
- Retrieve Method : Hybrid (Weight 0.7 Dense : 0.3 BM25)
- Reranker Model : Qwen3-Reranker-4B
- Generate Model : THaLLE-0.2-ThaiLLM-8b-fa

**Note** มีการศึกษาเทคนิกเพิ่มเติมสำหรับการ RAG จาก ***github.com/NirDiamant/RAG_Techniques*** และพิจารณาเลือกเทคนิก #11 Relevant Segment Extraction มาใช้ร่วมด้วย

- source: https://github.com/NirDiamant/RAG_Techniques

# FahMai RAG — Thai LLM 8B Solution

ระบบ **Retrieval-Augmented Generation (RAG)** สำหรับ FahMai Electronics Q&A

**Sections:**
- **Section 0:** Setup
- **Section 1:** Load Knowledge Base & Preprocessing
- **Section 2:** Dense Retrieval (QWEN3)
- **Section 3:** Sparse Retrieval (BM25)
- **Section 4:** Hybrid Retrieval (RRF)
- **Section 5:** Reranking (QWEN3)
- **Section 5.5:** Relevant Segment Extraction Retrieving
- **Section 6:** Prompt Design & Generation
- **Section 7:** Results & Submission

Note: โค้ดนี้รันบนคลาวด์ GPU ผ่าน Platform www.Vast.AI โดยกำหนด GPU:0 ให้แก่ LLM THaLLE-0.2-ThaiLLM-8b-fa และกำหนด GPU:1 ให้กับ Qwen3-4B Embedding และ Reranker


# Section 0: Setup

## ดาวโหลด Dependencies สำหรับการใช้งานแต่ละโมเดลผ่าน Huggingface

In [ ]:
import sys

# ===== DEPENDENCIES =====

!{sys.executable} -m pip install -q --root-user-action=ignore \
    "transformers>=4.51" \
    "sentence-transformers>=3.0.0" \
    pythainlp \
    rank-bm25 \
    python-dotenv \
    tqdm \
    accelerate \
    numpy

print("All dependencies installed successfully.")

All dependencies installed successfully.


In [ ]:
import numpy as np
from pathlib import Path
import requests, time, re, csv, os
print("ok: requests =", requests.__version__)

ok: requests = 2.32.5


## Config สำหรับการทดลอง และ การรันจริง

In [ ]:
# === CONFIGURATION ===
# Local inference (no external API)

N_QUESTIONS = 100   # 10 for demo, 100 for real submission
DATA_DIR    = "/workspace/Fahmai/data/"
KB_DIR      = f"{DATA_DIR}/knowledge_base"
MODEL_ID    = "KBTG-Labs/THaLLE-0.2-ThaiLLM-8B-fa"  # Local model on GPU

# ── Retrieval ──────────────────────────────────────────────────────────────────
RETRIEVAL_METHOD = "hybrid_rerank"   # "dense" | "bm25" | "hybrid" | "hybrid_rerank"

# ── RETRIEVAL_LEGS: กำหนดว่าจะใช้ retriever ตัวไหนบ้าง ─────────────────────
#   ["dense"]         → ใช้ Dense อย่างเดียว (ไม่สน weight)
#   ["bm25"]          → ใช้ BM25 อย่างเดียว (ไม่สน weight)
#   ["dense", "bm25"] → ใช้ทั้งคู่ (weight ถูกนำมาใช้ตามค่าด้านล่าง)
RETRIEVAL_LEGS = ["dense", "bm25"]  # ← เปลี่ยนตรงนี้

# Weights — ใช้เฉพาะเมื่อ RETRIEVAL_LEGS มีมากกว่า 1 ตัว (ต้อง sum = 1.0)
DENSE_WEIGHT = 0.7      # weight for Dense (Qwen3-Embedding)
BM25_WEIGHT  = 0.3      # weight for BM25 (word-level)

TOP_K          = 10       # number of chunks returned to LLM
RERANK_FETCH_K = 100      # number of chunks to fetch before reranking

CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 300

# Batch ส่ง LLM
LLM_BATCH_SIZE    = 4   # ล่ง Prompt Pararell
RETRIEVAL_WORKERS = 4   # Pararell CPU สำหรับ Retrieve


### โหลดโมเดล LLM สำหรับ Generate KBTG-0.2Thalle จาก HuggingFace

In [ ]:
# === LOCAL MODEL LOADING (ThaiLLM/ThaiLLM-8B) ===
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

print("Loading model:", MODEL_ID)
# use_fast=False — avoids "ModelWrapper" Rust deserialization error on ThaiLLM tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": "cuda:0"},
    trust_remote_code=True,
)

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=2048,   # ทดลองหลายค่าตั้งแต่ 512/1021/2048
    temperature=0.0,
    do_sample=False,
    return_full_text=False,
)
print("Model loaded and ready.")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: KBTG-Labs/THaLLE-0.2-ThaiLLM-8B-fa


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.52it/s]
Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded and ready.


### ดึงคำถาม

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS} for this run")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in questions[0]["choices"].items():
    print(f"  {k}. {v}")

Loaded 100 questions, using first 100 for this run

Example — Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  1. 3 ATM
  2. IP68
  3. 5 ATM
  4. IP67
  5. 10 ATM
  6. 20 ATM
  7. IPX8
  8. 1 ATM
  9. ไม่มีข้อมูลนี้ในฐานข้อมูล
  10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


---
# Section 1: ดึง Knowledges , Chunking และ Embedding

In [ ]:
# ดึงเอกสาร
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    rel = fp.relative_to(kb_dir)
    category = rel.parts[0] if len(rel.parts) > 1 else "store_info"
    documents.append({"path": str(rel), "text": text, "category": category})

# แสดงผลการดึงเอกสาร
print(f"Loaded {len(documents)} documents")
print(f"  products/:   {sum(1 for d in documents if d['category'] == 'products')}")
print(f"  policies/:   {sum(1 for d in documents if d['category'] == 'policies')}")
print(f"  store_info/: {sum(1 for d in documents if d['category'] == 'store_info')}")
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

Loaded 118 documents
  products/:   110
  policies/:   5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


### Chunking Strategy

- **Policy files** (ยาวมาก ~12–18 KB): ตัดตาม `## ` Markdown heading — แต่ละ section = 1 chunk
- **Product & Store files**: fixed-size window 512 chars with overlap 128 chars
- เพิ่ม metadata header ให้ทุก chunk เพื่อช่วย retrieval

In [ ]:
# function สำหรับการตัดข้อความเป็นก้อนๆ (chunk)
def make_chunks(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    # ถ้าข้อความสั้นอยู่แล้ว ก็ไม่ต้องตัด แค่คืนทั้งก้อนกลับไป
    if len(text) <= size:
        return [(text, 0, len(text))]  # (ข้อความ, จุดเริ่ม, จุดจบ)

    chunks = []
    start = 0

    # วนตัดข้อความไปเรื่อยๆ ทีละช่วง
    while start < len(text):
        end = min(start + size, len(text))  # กันไม่ให้เลยขอบข้อความ

        # เก็บข้อความช่วงนี้ + ตำแหน่ง
        chunks.append((text[start:end], start, end))

        # ขยับไปช่วงถัดไป (มี overlap คือให้ซ้ำกันเล็กน้อย)
        start += size - overlap

    return chunks


# function สำหรับตัดข้อความตามหัวข้อ (## ใน markdown)
def split_by_heading(text):
    # ตัดเอกสารตามหัวข้อ ## เพื่อให้แต่ละ section แยกกัน
    sections = []

    # split โดยใช้ regex → แยกตรงที่ขึ้นบรรทัดใหม่แล้วตามด้วย ##
    parts = re.split(r'\n(?=## )', text)

    offset = 0  # ใช้ตามหาตำแหน่งจริงใน text

    for part in parts:
        stripped = part.strip()

        # ถ้ามีข้อความจริง (ไม่ใช่ช่องว่าง)
        if stripped:
            # หาตำแหน่งเริ่มใน text จริง
            actual_start = text.find(stripped[:50], offset)

            # ถ้าหาไม่เจอ ใช้ offset แทน
            if actual_start == -1:
                actual_start = offset

            actual_end = actual_start + len(stripped)

            # เก็บ section พร้อมตำแหน่ง
            sections.append((stripped, actual_start, actual_end))

            # อัปเดต offset ไปท้าย section นี้
            offset = actual_end

    return sections


# เตรียม list สำหรับเก็บ chunk ทั้งหมด
chunks = []

# เก็บ text ต้นฉบับของแต่ละไฟล์ (เอาไว้ใช้ตอนอ้างอิงทีหลัง)
doc_text_map = {}  # path → text จริง


# วนทุก document ที่มี
for doc in documents:
    path     = doc['path']       # path ของไฟล์
    category = doc['category']   # ประเภทเอกสาร
    filename = Path(path).stem   # เอาแค่ชื่อไฟล์

    # สร้าง label อ่านง่ายจากชื่อไฟล์
    label = filename.split('_', 2)[-1].replace('_', ' ') if '_' in filename else filename

    # header เอาไว้แปะหน้าข้อความ (ช่วยให้รู้ว่า chunk นี้มาจากไหน)
    header = f'[{category} | {label}]\n'

    # เก็บ text เต็มไว้ก่อน (ใช้ตอน retrieve ย้อนกลับ)
    doc_text_map[path] = doc['text']

    doc_chunk_idx = 0  # ลำดับ chunk ในแต่ละไฟล์

    # ถ้าเป็น policy → ตัดตามหัวข้อ (##)
    if category == 'policies':
        for sec_text, char_s, char_e in split_by_heading(doc['text']):
            chunks.append({
                'text': header + sec_text,   # ข้อความ + header
                'source': path,              # มาจากไฟล์ไหน
                'doc_chunk_idx': doc_chunk_idx,  # chunk ลำดับที่เท่าไรในไฟล์นี้
                'char_start': char_s,        # ตำแหน่งเริ่มในไฟล์จริง
                'char_end': char_e,          # ตำแหน่งจบ
            })
            doc_chunk_idx += 1

    # ถ้าไม่ใช่ policy → ตัดแบบช่วงยาวเท่าๆ กัน
    else:
        for window_text, char_s, char_e in make_chunks(doc['text']):
            chunks.append({
                'text': header + window_text,
                'source': path,
                'doc_chunk_idx': doc_chunk_idx,
                'char_start': char_s,
                'char_end': char_e,
            })
            doc_chunk_idx += 1


# สรุปผลว่าตัดได้กี่ chunk
print(f'Created {len(chunks)} chunks from {len(documents)} documents')

# บอกว่ามี text ต้นฉบับเก็บไว้กี่ไฟล์
print(f'doc_text_map: {len(doc_text_map)} sources stored for RSE')

# แสดงตัวอย่าง chunk แรก
print(f'\n--- Sample chunk ---')
print(f"Source: {chunks[0]['source']}")
print(f"doc_chunk_idx: {chunks[0]['doc_chunk_idx']}, char_start: {chunks[0]['char_start']}, char_end: {chunks[0]['char_end']}")

# แสดงข้อความบางส่วน (300 ตัวแรก)
print(chunks[0]['text'][:300])

Created 603 chunks from 118 documents
doc_text_map: 118 sources stored for RSE

--- Sample chunk ---
Source: policies/cancellation_policy.md
doc_chunk_idx: 0, char_start: 0, char_end: 79
[policies | policy]
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---


---
# Section 2: Dense Retrieval (Qwen3-Embedding-4B')

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# โหลดโมเดลสำหรับแปลงข้อความเป็นตัวเลข (embedding)
# เอาไว้ใช้เทียบความใกล้เคียงของข้อความ
model_id = 'Qwen/Qwen3-Embedding-4B'

embed_model = SentenceTransformer(
    model_id,

    # ใช้ GPU ตัวที่ 1
    device='cuda:1',

    # ใช้ float16 เพื่อให้เร็วขึ้นและกินเมมน้อยลง
    model_kwargs={"torch_dtype": torch.float16},

    # padding ด้านซ้ายตามที่โมเดลต้องการ
    tokenizer_kwargs={"padding_side": "left"},

    # อนุญาตให้ใช้โค้ดเฉพาะของโมเดล
    trust_remote_code=True,
)

# เอาเฉพาะข้อความจากแต่ละ chunk
chunk_texts = [c['text'] for c in chunks]

# แปลงข้อความทั้งหมดเป็น embedding
chunk_embeddings = embed_model.encode(
    chunk_texts,

    # ทำทีละ 2 ข้อความ เพราะโมเดลค่อนข้างใหญ่
    batch_size=2,

    # ปรับค่า vector ให้พร้อมสำหรับการเทียบความใกล้เคียง
    normalize_embeddings=True,

    # แสดงสถานะตอนรัน
    show_progress_bar=True,

    # แปลงผลลัพธ์เป็น numpy array
    convert_to_numpy=True,
)

# ดูขนาดผลลัพธ์ (จำนวน chunk, ขนาด vector)
print(f"Embeddings shape: {chunk_embeddings.shape}")

`torch_dtype` is deprecated! Use `dtype` instead!
Batches: 100%|██████████| 302/302 [00:18<00:00, 16.41it/s]

Embeddings shape: (603, 2560)


In [ ]:
print(chunk_embeddings.shape)

(603, 2560)


### แสดงผลการ Retrieve

In [ ]:
def dense_retrieve(query, chunk_embs, k=TOP_K):
    # ฟังก์ชันนี้เอาไว้หา "ข้อความที่เกี่ยวข้องที่สุด"
    # โดยเอาคำถามไปเทียบกับ embedding ของทุก chunk

    q_emb = embed_model.encode(
        [query],

        # ปรับ vector ให้อยู่ในรูปแบบที่เทียบกันได้ง่าย
        normalize_embeddings=True,

        # แปลงเป็น numpy เพื่อคำนวณต่อ
        convert_to_numpy=True,

        # บอกโมเดลว่านี่คือ query (สำคัญสำหรับ Qwen)
        prompt_name="query",
    )

    # เอา embedding ของ query ไปเทียบกับทุก chunk
    # ค่ายิ่งสูง = ยิ่งใกล้เคียง
    scores = np.dot(chunk_embs, q_emb.T).flatten()

    # เรียงลำดับจากมาก → น้อย แล้วเลือกมา k อันดับแรก
    top_idx = np.argsort(scores)[::-1][:k]

    # คืนค่า index ของ chunk ที่ดีที่สุด + score
    return top_idx, scores[top_idx]


# ทดลองใช้กับคำถามข้อแรก
q = questions[0]

# หา chunk ที่เกี่ยวข้องที่สุด
idx, scores = dense_retrieve(q["question"], chunk_embeddings)

print(f"Question: {q['question']}\n")

# แสดงผลลัพธ์ทีละอันดับ
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")

    # แสดงข้อความบางส่วนของ chunk นั้น
    print(chunks[i]['text'][:200])
    print()

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Rank 1 (score=0.793) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
[products | watch s3 ultra]
 มิถุนายน 2569)

ตัวอย่าง: ฿14,990 ÷ 6 เดือน = เพียง ฿2,498.33/เดือน

---

## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้

**Q: Watch S3 Ultra ต่างจาก Watch S3 Pro ยังไงบ้างครับ?**
A:

Rank 2 (score=0.790) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
[products | watch s3 ultra]
ินระดับมาตรฐาน (เช่น ดำน้ำลึกเกิน 40 เมตร หรือดำน้ำในน้ำร้อน/น้ำพุร้อน)
- การดัดแปลงหรือซ่อมแซมโดยบุคคลที่ไม่ได้รับอนุญาตจากวงโคจร
- รอยขีดข่วนจากการใช้งานปกติ
- การสึกหรอข

Rank 3 (score=0.786) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
[products | watch s3 ultra]
# วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra)

รหัสสินค้า: WK-SW-001
แบรนด์: วงโคจร (WongKhoJon) — แบรนด์ในเครือฟ้าใหม่
หมวดหมู่: สมาร์ทวอทช์
ราคา: ฿14,990
สถานะ: มีส

Rank 4 (score=0.781) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
[products | watch s3 ultra]
---|-----------|
| หน้าจอ | 1.9 นิ

---
# Section 3: Sparse Retrieval (BM25)


In [ ]:
from pythainlp.tokenize import word_tokenize

# ข้อความตัวอย่าง (ภาษาไทยปนอังกฤษ)
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"

# ตัดคำภาษาไทยออกมาเป็นคำๆ
tokens = word_tokenize(sample, engine="newmm")

print(f"Input:  {sample}")

# แสดงผลลัพธ์ว่าถูกแยกเป็นคำอะไรบ้าง
print(f"Tokens: {tokens}")

Input:  Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens: ['Watch', ' ', 'S', '3', ' ', 'Ultra', ' ', 'กันน้ำ', 'ได้', 'กี่', ' ', 'ATM', ' ', 'ครับ', '?']


### Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

# ตัดคำของแต่ละ chunk ก่อน (เพราะ BM25 ทำงานกับคำเป็นหลัก)
tokenized_chunks = [
    word_tokenize(c["text"], engine="newmm")
    for c in chunks
]

# สร้าง BM25 index
# เอาไว้ใช้ค้นหาว่าข้อความไหนมีคำตรงกับ query มากที่สุด
bm25 = BM25Okapi(tokenized_chunks)

# บอกว่าทำ index เสร็จแล้ว มีทั้งหมดกี่ chunk
print(f"BM25 index built over {len(tokenized_chunks)} chunks")

BM25 index built over 603 chunks


###  Retrieve และเปรียบเทียบกับวิธี Dense


In [ ]:
def bm25_retrieve(query, k=TOP_K):
    # ฟังก์ชันนี้เอาไว้หา chunk ที่เกี่ยวข้อง
    # โดยดูจาก "คำที่ตรงกัน" (ไม่ใช่ความหมาย)

    # ตัดคำจาก query ก่อน
    tokens = word_tokenize(query, engine="newmm")

    # ให้ BM25 คำนวณคะแนนว่าแต่ละ chunk ตรงกับคำมากแค่ไหน
    scores = bm25.get_scores(tokens)

    # เรียงจากคะแนนมาก → น้อย แล้วเลือกมา k อันดับแรก
    top_idx = np.argsort(scores)[::-1][:k]

    # คืนค่า index ของ chunk + score
    return top_idx, scores[top_idx]


# ทดลองกับคำถามเดียวกัน เพื่อเทียบ 2 วิธี
q = questions[0]
print(f"Question: {q['question']}\n")

# วิธี embedding (เข้าใจความหมาย)
d_idx, _ = dense_retrieve(q["question"], chunk_embeddings)

# วิธี BM25 (ดูคำตรง)
b_idx, _ = bm25_retrieve(q["question"])


print("Dense top-5 sources:")
for i in d_idx:
    print(f"  {chunks[i]['source']}")

print("\nBM25 top-5 sources:")
for i in b_idx:
    print(f"  {chunks[i]['source']}")

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Dense top-5 sources:
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-004_wongkhojon_watch_s3_se.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-004_wongkhojon_watch_s3_se.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md

BM25 top-5 sources:
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_

---
# Section 4: Hybrid Retrieval (RRF)

ให้ค่าน้ำหนักของวิธี Dense ที่ 0.7 และค่าน้ำหนักของวิธ๊ BM 25 ที่ 0.3 โดยผลคะแนนคำนวณจาก

\begin{equation}
RRFscore(d) = \sum_{r \in R} w_r \cdot \frac{1}{k + r(d)}
\end{equation}


In [ ]:
def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60, max_per_source=3):
    # ฟังก์ชันนี้เอาไว้ค้นหาข้อมูลแบบผสม
    # คือใช้ได้ทั้งแบบดู "ความหมาย" และแบบดู "คำที่ตรงกัน"

    legs = RETRIEVAL_LEGS
    fetch_k = k * 4

    # ถ้าเลือกใช้แค่วิธีเดียว ก็เรียกวิธีนั้นตรงๆ
    if len(legs) == 1:
        if legs[0] == "dense":
            idx, _ = dense_retrieve(query, chunk_embs, k=k)
            return list(idx)
        elif legs[0] == "bm25":
            idx, _ = bm25_retrieve(query, k=k)
            return list(idx)

    # ถ้าใช้หลายวิธีพร้อมกัน
    # กำหนดน้ำหนักของแต่ละวิธีที่เปิดใช้อยู่
    active_weights = {}
    if "dense" in legs:
        active_weights["dense"] = DENSE_WEIGHT
    if "bm25" in legs:
        active_weights["bm25"] = BM25_WEIGHT

    # ปรับน้ำหนักให้รวมกันเป็น 1
    total = sum(active_weights.values())
    for key in active_weights:
        active_weights[key] /= total

    # เก็บคะแนนรวมของแต่ละ chunk
    rrf_scores = {}

    # เอาผลจาก dense มาใส่คะแนน
    # อันดับยิ่งดี ก็ยิ่งได้คะแนนมาก
    if "dense" in active_weights:
        w = active_weights["dense"]
        d_idx, _ = dense_retrieve(query, chunk_embs, k=fetch_k)
        for rank, idx in enumerate(d_idx, 1):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + w / (rrf_k + rank)

    # เอาผลจาก BM25 มาใส่คะแนนด้วย
    if "bm25" in active_weights:
        w = active_weights["bm25"]
        b_idx, _ = bm25_retrieve(query, k=fetch_k)
        for rank, idx in enumerate(b_idx, 1):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + w / (rrf_k + rank)

    # เรียง chunk ตามคะแนนรวม จากมากไปน้อย
    sorted_idx = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)

    # กระจายผลลัพธ์ ไม่ให้มาจากไฟล์เดียวซ้ำเยอะเกินไป
    source_count = {}
    deduped = []
    spillover = []

    for idx in sorted_idx:
        src = chunks[idx]["source"]
        cnt = source_count.get(src, 0)

        # ถ้าไฟล์นี้ยังไม่เกินจำนวนที่กำหนด ก็เอาไว้
        if cnt < max_per_source:
            source_count[src] = cnt + 1
            deduped.append(idx)
        else:
            # ถ้าเกินแล้ว เก็บไว้เผื่อสำรอง
            spillover.append(idx)

        # ถ้าได้ครบ k แล้วก็หยุด
        if len(deduped) == k:
            break

    # ถ้ายังไม่ครบ k ค่อยเติมจากตัวสำรอง
    if len(deduped) < k:
        deduped.extend(spillover[: k - len(deduped)])

    return deduped[:k]


# ทดลองเรียกใช้
q = questions[0]
print(f"Q1: {q['question']}")
print(f"Active legs: {RETRIEVAL_LEGS}\n")

h_idx = hybrid_retrieve(q["question"], chunk_embeddings)

# เอาแค่ชื่อไฟล์มาดูง่ายๆ
srcs = [chunks[i]["source"].split("/")[-1][:40] for i in h_idx]

print("Hybrid result:")
for r, s in enumerate(srcs, 1):
    print(f"  {r}. {s}")

Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
Active legs: ['dense', 'bm25']

Hybrid result:
  1. WK-SW-001_wongkhojon_watch_s3_ultra.md
  2. WK-SW-001_wongkhojon_watch_s3_ultra.md
  3. WK-SW-002_wongkhojon_watch_s3_pro.md
  4. WK-SW-001_wongkhojon_watch_s3_ultra.md
  5. WK-SW-002_wongkhojon_watch_s3_pro.md
  6. WK-SW-003_wongkhojon_watch_s3.md
  7. WK-SW-003_wongkhojon_watch_s3.md
  8. WK-SW-004_wongkhojon_watch_s3_se.md
  9. WK-SW-003_wongkhojon_watch_s3.md
  10. WK-SW-004_wongkhojon_watch_s3_se.md
  11. WK-SW-004_wongkhojon_watch_s3_se.md
  12. WK-SW-008_wongkhojon_watch_s2_pro.md
  13. WK-SW-008_wongkhojon_watch_s2_pro.md
  14. WK-SW-006_wongkhojon_watch_classic_c1.md
  15. WK-SW-002_wongkhojon_watch_s3_pro.md


---
# Section 5: Rerank โดยใช้โมเดล Qwen3-Reranker-4B

In [ ]:
from transformers import AutoModelForCausalLM as _AutoModelForCausalLM_Reranker
from transformers import AutoTokenizer as _AutoTokenizer_Reranker
import torch

print("Loading Reranker (Qwen3-Reranker-4B) ...")
RERANKER_MODEL_ID = "Qwen/Qwen3-Reranker-4B"

# โหลด tokenizer สำหรับแปลงข้อความเป็นตัวเลข
reranker_tokenizer = _AutoTokenizer_Reranker.from_pretrained(
    RERANKER_MODEL_ID,
    padding_side="left"
)

# โหลดโมเดล reranker
# โมเดลนี้เอาไว้ตัดสินว่า "ข้อความนี้ตอบคำถามได้ไหม"
reranker_model = _AutoModelForCausalLM_Reranker.from_pretrained(
    RERANKER_MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": "cuda:1"},   # เอาโมเดลไปไว้ที่ GPU
).eval()

# เตรียม token ของคำว่า yes / no
# โมเดลจะตอบออกมาเป็นแนวนี้ เราเลยดึงค่าออกมาวัดคะแนน
token_false_id = reranker_tokenizer.convert_tokens_to_ids("no")
token_true_id  = reranker_tokenizer.convert_tokens_to_ids("yes")

# ความยาว input สูงสุดที่โมเดลรับได้
reranker_max_length = 8192

# ข้อความนำหน้า (บอกโมเดลว่าต้องทำอะไร)
reranker_prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"

# ข้อความปิดท้าย (format ที่โมเดลคาดหวัง)
reranker_suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

# แปลง prefix/suffix เป็น token
prefix_tokens = reranker_tokenizer.encode(reranker_prefix, add_special_tokens=False)
suffix_tokens = reranker_tokenizer.encode(reranker_suffix, add_special_tokens=False)


# สร้างข้อความที่รวม query + document เข้าด้วยกัน
def format_instruction(instruction, query, doc):
    # ถ้าไม่ได้กำหนด instruction ก็ใช้ค่า default
    if instruction is None:
        instruction = "Given a web search query, retrieve relevant passages that answer the query"

    return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}"


# แปลงข้อความทั้งหมดเป็น input ที่โมเดลเข้าใจ
def process_reranker_inputs(pairs):
    # tokenize ข้อความ
    inputs = reranker_tokenizer(
        pairs,
        padding=False,
        truncation="longest_first",
        return_attention_mask=False,
        max_length=reranker_max_length - len(prefix_tokens) - len(suffix_tokens)
    )

    # ใส่ prefix และ suffix เข้าไป
    for i, ele in enumerate(inputs["input_ids"]):
        inputs["input_ids"][i] = prefix_tokens + ele + suffix_tokens

    # เติม padding ให้ยาวเท่ากัน และแปลงเป็น tensor
    inputs = reranker_tokenizer.pad(
        inputs,
        padding=True,
        return_tensors="pt",
        max_length=reranker_max_length
    )

    # ย้ายไป GPU
    for key in inputs:
        inputs[key] = inputs[key].to(reranker_model.device)

    return inputs


# คำนวณคะแนนว่า document นี้ "ใช่คำตอบไหม"
@torch.no_grad()
def compute_reranker_logits(inputs):
    # เอาผลลัพธ์จากโมเดล (ตำแหน่งสุดท้าย)
    batch_scores = reranker_model(**inputs).logits[:, -1, :]

    # ดึงค่าเฉพาะ yes / no
    true_vector  = batch_scores[:, token_true_id]
    false_vector = batch_scores[:, token_false_id]

    # รวมเป็น [no, yes]
    batch_scores = torch.stack([false_vector, true_vector], dim=1)

    # แปลงเป็น probability
    batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)

    # เอาเฉพาะความน่าจะเป็นของ yes
    scores = batch_scores[:, 1].exp().tolist()

    return scores


print(f"Reranker loaded on: {reranker_model.device}")
print(f"  token_true_id (yes) = {token_true_id}")
print(f"  token_false_id (no) = {token_false_id}")

Loading Reranker (Qwen3-Reranker-4B) ...


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.54it/s]


Reranker loaded on: cuda:1
  token_true_id (yes) = 9693
  token_false_id (no) = 2152


In [ ]:
RERANKER_TASK = "จากคำถามเกี่ยวกับสินค้าอิเล็กทรอนิกส์ร้านฟ้าใหม่ ค้นหาข้อมูลที่ตรงกับคำถามมากที่สุด"
RERANKER_BATCH_SIZE = 2  # ทำทีละกี่คู่ข้อความ

def rerank_chunks(query, chunk_indices, top_k=TOP_K):
    # ฟังก์ชันนี้เอาไว้คัดรอบสุดท้าย
    # จาก chunk ที่หาเจอมาแล้ว ว่าอันไหนตรงคำถามที่สุด

    if not chunk_indices:
        return []

    # เอาคำถามไปจับคู่กับ chunk ทีละอัน
    pairs = [
        format_instruction(RERANKER_TASK, query, chunks[idx]["text"])
        for idx in chunk_indices
    ]

    all_scores = []

    # ส่งเข้าโมเดลทีละ batch
    for start in range(0, len(pairs), RERANKER_BATCH_SIZE):
        batch_pairs = pairs[start : start + RERANKER_BATCH_SIZE]

        # เตรียม input ให้พร้อมสำหรับโมเดล
        inputs = process_reranker_inputs(batch_pairs)

        # ให้โมเดลให้คะแนนว่า chunk นี้ตรงคำถามแค่ไหน
        batch_scores = compute_reranker_logits(inputs)

        # เก็บคะแนนทั้งหมดไว้รวมกัน
        all_scores.extend(batch_scores)

    # จับคู่ index ของ chunk กับคะแนนที่ได้
    scored_indices = list(zip(chunk_indices, all_scores))

    # เรียงจากคะแนนมากไปน้อย
    scored_indices.sort(key=lambda x: x[1], reverse=True)

    # คืนเฉพาะ chunk ที่ดีที่สุดตามจำนวนที่ต้องการ
    return [idx for idx, score in scored_indices[:top_k]]


# ทดลองใช้งาน
q = questions[0]
print(f"Q1: {q['question']}\n")

# หา chunk มาก่อนรอบแรก
pool_indices = hybrid_retrieve(q["question"], chunk_embeddings, k=RERANK_FETCH_K)

# แล้วค่อยให้ reranker ช่วยจัดอันดับใหม่
reranked_indices = rerank_chunks(q["question"], pool_indices, top_k=TOP_K)

print(f"Hybrid (Pool of {RERANK_FETCH_K}) -> Reranked Top {TOP_K}:")
for r, i in enumerate(reranked_indices, 1):
    src = chunks[i]["source"].split("/")[-1][:40]
    print(f"  {r}. {src}")

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ



/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2919: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Hybrid (Pool of 100) -> Reranked Top 15:
  1. WK-SW-001_wongkhojon_watch_s3_ultra.md
  2. WK-SW-001_wongkhojon_watch_s3_ultra.md
  3. WK-SW-002_wongkhojon_watch_s3_pro.md
  4. WK-SW-003_wongkhojon_watch_s3.md
  5. WK-SW-004_wongkhojon_watch_s3_se.md
  6. WK-SW-002_wongkhojon_watch_s3_pro.md
  7. WK-SW-001_wongkhojon_watch_s3_ultra.md
  8. WK-SW-002_wongkhojon_watch_s3_pro.md
  9. WK-SW-008_wongkhojon_watch_s2_pro.md
  10. WK-SW-004_wongkhojon_watch_s3_se.md
  11. WK-SW-006_wongkhojon_watch_classic_c1.md
  12. WK-SW-003_wongkhojon_watch_s3.md
  13. WK-SW-008_wongkhojon_watch_s2_pro.md
  14. WK-SW-008_wongkhojon_watch_s2_pro.md
  15. WK-SW-004_wongkhojon_watch_s3_se.md


---
# Section 5.5: Relevant Segment Extraction (RSE)

นำ Chunk ที่ได้จากการ Retrieve มาต่อกันมาต่อกันก่อนส่งให้ LLM

Source: https://github.com/NirDiamant/RAG_Techniques/tree/main/all_rag_techniques


In [ ]:
from collections import defaultdict

def rse_merge(chunk_indices, max_gap=1):
    # ฟังก์ชันนี้เอาไว้รวม chunk ที่อยู่ใกล้กันในไฟล์เดียวกัน
    # เพื่อให้ได้ข้อความต่อเนื่อง อ่านง่ายกว่าแยกเป็นก้อนๆ

    if not chunk_indices:
        return []

    # แยก chunk ตาม source ก่อน
    source_groups = defaultdict(list)
    for idx in chunk_indices:
        c = chunks[idx]
        source_groups[c['source']].append(idx)

    # หา run ของ chunk ที่อยู่ติดกันหรือห่างกันไม่มาก
    segments = []  # จะเก็บเป็น (source, [idx1, idx2, ...])
    for src, indices in source_groups.items():
        # เรียงตามตำแหน่งเดิมในเอกสาร
        indices.sort(key=lambda i: chunks[i]['doc_chunk_idx'])

        # เริ่มจับกลุ่ม chunk ที่ติดกัน
        current_run = [indices[0]]
        for j in range(1, len(indices)):
            prev_pos = chunks[current_run[-1]]['doc_chunk_idx']
            curr_pos = chunks[indices[j]]['doc_chunk_idx']

            # ถ้ายังอยู่ใกล้กัน ก็รวมกลุ่มเดียวกัน
            if curr_pos - prev_pos <= max_gap + 1:
                current_run.append(indices[j])
            else:
                # ถ้าเริ่มห่างกันแล้ว ปิดกลุ่มเดิมก่อน
                segments.append((src, current_run))
                current_run = [indices[j]]

        # เก็บกลุ่มสุดท้าย
        segments.append((src, current_run))

    merged_results = []

    # เอากลุ่มที่ได้มารวมเป็นข้อความจริง
    for src, run_indices in segments:
        # เตรียม header เอาไว้บอกว่า segment นี้มาจากไหน
        category = src.split('/')[0] if '/' in src else 'store_info'
        filename = Path(src).stem
        label = filename.split('_', 2)[-1].replace('_', ' ') if '_' in filename else filename
        header = f'[{category} | {label}]\n'

        # ถ้ามีแค่ chunk เดียว ก็ใช้ข้อความเดิมได้เลย
        if len(run_indices) == 1:
            merged_results.append({
                'text': chunks[run_indices[0]]['text'],
                'source': src,
                'chunk_count': 1,
            })
        else:
            # ถ้ามีหลาย chunk ที่ต่อกัน
            # จะตัดจาก text ต้นฉบับตรงๆ เพื่อไม่ให้มี overlap ซ้ำ
            char_start = chunks[run_indices[0]]['char_start']
            char_end   = chunks[run_indices[-1]]['char_end']

            if src in doc_text_map:
                original = doc_text_map[src]
                segment_text = original[char_start:char_end]

                merged_results.append({
                    'text': header + segment_text,
                    'source': src,
                    'chunk_count': len(run_indices),
                })
            else:
                # กรณีหา text ต้นฉบับไม่เจอ ก็คืนทีละ chunk แทน
                for idx in run_indices:
                    merged_results.append({
                        'text': chunks[idx]['text'],
                        'source': src,
                        'chunk_count': 1,
                    })

    # เรียงลำดับผลลัพธ์ให้ใกล้เคียงกับอันดับเดิมจาก rerank
    index_rank = {idx: rank for rank, idx in enumerate(chunk_indices)}

    def best_rank(seg):
        src = seg['source']

        # หา rank ที่ดีที่สุดของ source นี้
        relevant_indices = [i for i in chunk_indices if chunks[i]['source'] == src]
        return min(index_rank.get(i, 9999) for i in relevant_indices)

    merged_results.sort(key=best_rank)

    return merged_results


# ทดลองใช้กับคำถามข้อแรก
q = questions[0]
print(f'Q1: {q["question"]}\n')

# หา candidate มาก่อน
pool = hybrid_retrieve(q['question'], chunk_embeddings, k=RERANK_FETCH_K)

# rerank เพื่อคัด chunk ที่ตรงคำถามที่สุด
reranked = rerank_chunks(q['question'], pool, top_k=TOP_K)

# ดูผลก่อน merge
print('Before RSE')
for r, i in enumerate(reranked, 1):
    src = chunks[i]['source'].split('/')[-1][:40]
    idx_in_doc = chunks[i]['doc_chunk_idx']
    print(f'  {r}. {src} [chunk #{idx_in_doc}]')

# รวม chunk ที่อยู่ติดกัน
merged = rse_merge(reranked, max_gap=1)

# ดูผลหลัง merge
print(f'\nAfter RSE ({len(reranked)} chunks -> {len(merged)} segments)')
for r, seg in enumerate(merged, 1):
    src = seg['source'].split('/')[-1][:40]
    n = seg['chunk_count']
    txt_len = len(seg['text'])
    print(f'  {r}. {src} [{n} chunks merged, {txt_len} chars]')

Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

=== Before RSE (individual chunks) ===
  1. WK-SW-001_wongkhojon_watch_s3_ultra.md [chunk #0]
  2. WK-SW-001_wongkhojon_watch_s3_ultra.md [chunk #5]
  3. WK-SW-002_wongkhojon_watch_s3_pro.md [chunk #5]
  4. WK-SW-003_wongkhojon_watch_s3.md [chunk #1]
  5. WK-SW-004_wongkhojon_watch_s3_se.md [chunk #1]
  6. WK-SW-002_wongkhojon_watch_s3_pro.md [chunk #4]
  7. WK-SW-001_wongkhojon_watch_s3_ultra.md [chunk #4]
  8. WK-SW-002_wongkhojon_watch_s3_pro.md [chunk #2]
  9. WK-SW-008_wongkhojon_watch_s2_pro.md [chunk #1]
  10. WK-SW-004_wongkhojon_watch_s3_se.md [chunk #2]
  11. WK-SW-006_wongkhojon_watch_classic_c1.md [chunk #1]
  12. WK-SW-003_wongkhojon_watch_s3.md [chunk #0]
  13. WK-SW-008_wongkhojon_watch_s2_pro.md [chunk #2]
  14. WK-SW-008_wongkhojon_watch_s2_pro.md [chunk #0]
  15. WK-SW-004_wongkhojon_watch_s3_se.md [chunk #0]

=== After RSE (15 chunks → 7 segments) ===
  1. WK-SW-001_wongkhojon_watch_s3_ultra.md [1 chunks merged, 1028 chars]
 

---
# Section 6: Prompt Design & Generation

### 6.1 Prompt Structure และตัวอย่างผลลัพธ์การถามตอบ

In [ ]:
import re, time

def parse_answer(text):
    # ฟังก์ชันนี้เอาไว้ดึง "เลขคำตอบ" ออกจากข้อความที่โมเดลตอบกลับมา

    # ถ้าไม่มีข้อความเลย ก็กันพังไว้ก่อน
    if text is None:
        return 1

    # ลบส่วน <think> ออก เพื่อเอาเฉพาะคำตอบจริง
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    clean = re.sub(r"<think>.*$", "", clean, flags=re.DOTALL).strip()

    # กรณีปกติที่สุด เช่น ANSWER: 4
    m = re.search(r"ANSWER:\s*(\d{1,2})", clean, re.IGNORECASE)
    if m:
        val = int(m.group(1))
        if 1 <= val <= 10:
            return val

    # ถ้ารูปแบบไม่ตรง แต่มีเลขอยู่ท้ายข้อความ ก็ลองดึงจากตรงนั้น
    m = re.search(r"(?:^|\n)\s*(\d{1,2})\s*$", clean)
    if m:
        val = int(m.group(1))
        if 1 <= val <= 10:
            return val

    # เผื่อบางที ANSWER: X ไปอยู่ใน text เดิมที่ยังมี <think>
    m = re.search(r"ANSWER:\s*(\d{1,2})", text, re.IGNORECASE)
    if m:
        val = int(m.group(1))
        if 1 <= val <= 10:
            return val

    # ถ้าหาไม่เจอจริงๆ ค่อยคืน None
    return None


# ข้อความตั้งต้นที่บอกโมเดลว่าต้องทำหน้าที่อะไร
SYSTEM_PROMPT = (
    "คุณเป็น AI ผู้ช่วยร้านฟ้าใหม่ Electronics \n"
    "คุณมีความสามารถอย่างมากในการให้ตอบด้านสินค้าอิเล็กโทรนิกส์ เช่น สเปคคอมพิวเตอร์ \n"
    "หน้าที่ของคุณคุณตอบคำถามทั่วไปที่ลูกค้าถามมาจากข้อมูลทีมีเท่านั้น \n"
    "ร้านมีเฉพาะสินค้าของแบรนด์ดังต่อไปนี้เท่านั้น:\n"
    "  แบรนด์ในเครือฟ้าใหม่: สายฟ้า (SaiFah), ดาวเหนือ (DaoNuea), คลื่นเสียง (KluenSiang), วงโคจร (WongKhoJon), จุดเชื่อม (JudChuam)\n"
    "  แบรนด์พันธมิตร: อาร์คเวฟ (ArcWave), เซนไบต์ (ZenByte), พัลส์เกียร์ (PulseGear), โนวาเทค (NovaTech)\n"
    "สินค้าแบรนด์อื่นนอกเหนือ 9 แบรนด์ข้างต้น ไม่มีขายในร้าน\n\n"
    "หากเป็นสินค้าด้าน spec หรือคำถาม Technical ให้คิดวิเคราะห์จากข้อมูลที่มี เช่น \n"
    "spec คอมบอกว่า RAM เป็นแบบ SO-DIMM = เพิ่ม RAM ภายหลังได้ \n"
    "กฎการตอบ:\n"
    "  1. อ่านข้อมูลที่ให้มาอย่างละเอียด แล้วเลือกตัวเลือกที่ตรงกับข้อมูลมากที่สุด\n"
    "  2. ถ้าคำถามเกี่ยวกับร้านฟ้าใหม่ สินค้า 9 แบรนด์ หรือ คำถามเกี่ยวกับร้านฟ้าใหม่ หรือนโยบายร้านฟ้าใหม่ แต่ไม่พบคำตอบในข้อมูลที่ให้มา → ตอบ 9\n"
    "  3. ถ้าคำถามไม่เกี่ยวข้องกับร้านฟ้าใหม่หรือสินค้าอิเล็กทรอนิกส์เลย (เช่น สูตรอาหาร วันหยุดราชการ ดอกเบี้ยเงินฝาก ตั๋วเครื่องบิน) → ตอบ 10\n\n"
    "  4. คำถามเปรียบเทียบสินค้าที่มีขายในร้าน ให้ตรวจข้อมูลของสินค้าที่กล่าวถึง แล้วเปรียบเทียบตามข้อเท็จจริงในข้อมูล\n"
    "ตัวอย่างการตอบ:\n"
    "  - ถามสเปคสินค้าที่มีในข้อมูล เช่น ความจุแบตเตอรี่ → เลือกตัวเลือกที่ตรงกับข้อมูล → ANSWER: เลือกตัวเลือก 1 ถึง 8 ที่ถูกต้องที่สุด\n"
    "  - ถามข้อมูลสินค้าที่ไม่มีระบุในข้อมูล เช่น จำนวนพนักงาน → ANSWER: 9\n"
    "  หรือ ถามว่า \"สินค้า Y ผลิตจากประเทศอะไร\" + ในข้อมูลไม่ได้ระบุประเทศผลิต → ANSWER: 9\n"
    "  หรือ สถานการณ์ที่ไม่มีข้อมูลเพียงพอในการตอบ choice 1-8 เช่น ถามว่า \"สินค้า Z เล่มเกม ... ได้ไหม หรือ เอาไปทำ ... ได้ไหม \n"
    "  + ไม่มีข้อมูลเกี่ยวกับการนำสินค้าในไปใช้เรื่องนั้นเลย + ใน choices 1-8 ไม่มีข้อใดถูกต้อง → ANSWER: 9\n"
    "  - ถามเรื่องที่ไม่เกี่ยวกับร้านฟ้าใหม่หรือสินค้าอิเล็กทรอนิกส์เลย เช่น สภาพอากาศวันนี้ หรือ ระยะทางจากจังหวัด A ไปจังหวัด B → ANSWER: 10\n\n"
    "คุณต้องเลือกตัวเลือกมาตอบเสมอ\n"
    "ตอบในรูปแบบ ANSWER: X เท่านั้น (X คือตัวเลข 1-10)"
)

# สร้างข้อความฝั่ง user
# เอาข้อมูลที่ค้นเจอ + คำถาม + ตัวเลือก มารวมกัน
def build_rag_prompt(question, choices, retrieved_chunks):
    context = "\n\n".join(
        f"=== ข้อมูล {i+1} [{c['source']}] ===\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )

    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())

    return (
        f"{context}\n\n"
        f"คำถาม: {question}\n\n"
        f"ตัวเลือก:\n{choices_text}"
    )

# เอา system กับ user message ไปใส่ในรูปแบบที่โมเดลต้องการ
def apply_chat(system_msg, user_msg):
    msgs = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg}
    ]

    return tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True,
    )

# ฟังก์ชันถามโมเดล
# ทำครบตั้งแต่สร้าง prompt จนได้ข้อความตอบกลับ
def ask_llm(question, choices, retrieved_chunks, delay_sec=0.5):
    rag = build_rag_prompt(question, choices, retrieved_chunks)
    prompt = apply_chat(SYSTEM_PROMPT, rag)

    out = gen(prompt)[0]["generated_text"].strip()

    # หน่วงนิดหน่อย เผื่อใช้ตอน demo
    time.sleep(delay_sec)

    return out

# ทดลองถามข้อแรก
q = questions[0]

# ดึงข้อมูลที่น่าจะเกี่ยวข้องมาก่อน
h_idx = hybrid_retrieve(q["question"], chunk_embeddings)
retrieved = [chunks[i] for i in h_idx]

# ส่งให้โมเดลตอบ
raw = ask_llm(q["question"], q["choices"], retrieved)

# แปลงข้อความตอบให้เหลือแค่เลขคำตอบ
answer = parse_answer(raw)

print(f"Q{q['id']}: {q['question']}")
print(f"LLM raw:       {raw[:100]}... (truncated)")
print(f"Parsed answer: {answer}")

Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
LLM raw:       <think>
Okay, let's tackle this question. The user is asking about the water resistance rating of th... (truncated)
Parsed answer: 5


### 6.2 Run จริง

In [ ]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

def build_full_prompt(q):
    # ฟังก์ชันนี้ทำงานให้ครบสำหรับ 1 คำถาม
    # คือหา context ที่เกี่ยวข้อง แล้วประกอบเป็น prompt พร้อมส่งเข้าโมเดล

    # ถ้าเลือกใช้ dense อย่างเดียว
    if RETRIEVAL_METHOD == "dense":
        idxs, _ = dense_retrieve(q["question"], chunk_embeddings)
        retrieved = [chunks[i] for i in idxs]

    # ถ้าเลือกใช้ BM25 อย่างเดียว
    elif RETRIEVAL_METHOD == "bm25":
        idxs, _ = bm25_retrieve(q["question"])
        retrieved = [chunks[i] for i in idxs]

    # ถ้าเลือกใช้ hybrid แต่ยังไม่ rerank
    elif RETRIEVAL_METHOD == "hybrid":
        idxs = hybrid_retrieve(q["question"], chunk_embeddings)
        retrieved = [chunks[i] for i in idxs]

    # ถ้าเลือกแบบเต็ม คือ hybrid + rerank + รวม chunk ที่ติดกัน
    else:
        pool = hybrid_retrieve(q["question"], chunk_embeddings, k=RERANK_FETCH_K)
        idxs = rerank_chunks(q["question"], pool, top_k=TOP_K)

        # รวม chunk ที่อยู่ติดกันให้กลายเป็นข้อความต่อเนื่อง
        retrieved = rse_merge(idxs, max_gap=1)

    # เอา context ที่หาได้มาสร้าง prompt
    rag = build_rag_prompt(q["question"], q["choices"], retrieved)

    # จัดให้อยู่ในรูปแบบ chat ที่โมเดลใช้
    full = apply_chat(SYSTEM_PROMPT, rag)

    # คืน id ของคำถามคู่กับ prompt ที่สร้างเสร็จแล้ว
    return q["id"], full


def run_pipeline(questions, n=N_QUESTIONS):
    # ฟังก์ชันหลักของทั้ง pipeline
    # ช่วงแรกเตรียม prompt
    # ช่วงหลังค่อยส่งเข้า LLM เพื่อให้ตอบ

    qs = questions[:n]

    # ช่วงแรก ทำ retrieval และสร้าง prompt หลายคำถามพร้อมกัน
    print(f"Phase 1 — retrieval + RSE [{RETRIEVAL_METHOD}]: "
          f"{n} questions × {RETRIEVAL_WORKERS} workers")
    print(f"  Active legs: {RETRIEVAL_LEGS}")

    # ถ้าใช้หลายวิธีร่วมกัน ก็แสดงน้ำหนักของแต่ละวิธี
    if len(RETRIEVAL_LEGS) > 1:
        print(f"  Weights → Dense: {DENSE_WEIGHT}, BM25: {BM25_WEIGHT}")
    else:
        print(f"  Single-leg mode → weights not used")

    print(f"  RSE: enabled (max_gap=1)")

    # ใช้หลาย worker ช่วยกันเตรียม prompt ให้เร็วขึ้น
    with ThreadPoolExecutor(max_workers=RETRIEVAL_WORKERS) as executor:
        prompt_pairs = list(tqdm(
            executor.map(build_full_prompt, qs),
            total=n,
            desc="Retrieval+RSE"
        ))

    # ก่อนส่งเข้าโมเดล ตั้ง padding ด้านซ้าย
    tokenizer.padding_side = "left"

    predictions = {}

    # ช่วงสอง ส่ง prompt เข้า LLM เป็น batch
    print(f"\nPhase 2 — LLM inference: batch_size={LLM_BATCH_SIZE}")
    for i in tqdm(range(0, n, LLM_BATCH_SIZE), desc="LLM Batches"):
        batch = prompt_pairs[i : i + LLM_BATCH_SIZE]

        # แยก id กับ prompt ออกจากกัน
        qids = [p[0] for p in batch]
        prompts = [p[1] for p in batch]

        # ให้โมเดลตอบทีละ batch
        outputs = gen(prompts, batch_size=LLM_BATCH_SIZE)

        # เอาคำตอบที่ได้มาแปลงเป็นเลข
        for qid, out in zip(qids, outputs):
            raw = out[0]["generated_text"].strip()
            pred = parse_answer(raw)

            # ถ้า parse ไม่ได้จริงๆ ให้ fallback เป็น 1
            predictions[qid] = pred if pred else 1

            # ทำข้อความสั้นๆ ไว้ดู log
            display_raw = raw.replace('\n', '\\n')
            if len(display_raw) > 50:
                display_raw = display_raw[:50] + "..."

            print(f"  Q{qid}: '{display_raw}' \u2192 {pred}")

    print(f"\nDone — answered {len(predictions)} / {n} questions")
    return predictions


hybrid_preds = run_pipeline(questions, n=N_QUESTIONS)

Phase 1 — retrieval + RSE [hybrid_rerank]: 100 questions × 4 workers
  Active legs: ['dense', 'bm25']
  Weights → Dense: 0.5, BM25: 0.5
  RSE: enabled (max_gap=1)


Retrieval+RSE: 100%|██████████| 100/100 [07:48<00:00,  4.68s/it]



Phase 2 — LLM inference: batch_size=4


LLM Batches:   4%|▍         | 1/25 [00:46<18:27, 46.14s/it]

  Q1: '<think>\nOkay, let's tackle this question. The use...' → 5
  Q2: '<think>\nOkay, let's tackle this question. The use...' → 7
  Q3: '<think>\nOkay, let's tackle this question. The use...' → 2
  Q4: '<think>\nOkay, let's tackle this question. The use...' → 6


...

  Q89: '<think>\nOkay, let's tackle this question. The use...' → 4
  Q90: '<think>\nOkay, let's tackle this question step by ...' → 3
  Q91: '<think>\nOkay, let's tackle this question. The use...' → 4
  Q92: '<think>\nOkay, let's tackle this question. The use...' → 6


LLM Batches:  96%|█████████▌| 24/25 [17:10<00:51, 51.99s/it]

  Q93: '<think>\nOkay, let's tackle this question. The use...' → 5
  Q94: '<think>\nOkay, let's tackle this question. The use...' → 4
  Q95: '<think>\nOkay, let's tackle this question. The use...' → 4
  Q96: '<think>\nOkay, let's tackle this question. The use...' → 5


LLM Batches: 100%|██████████| 25/25 [18:52<00:00, 45.31s/it]

  Q97: '<think>\nOkay, let's tackle this question. The use...' → 7
  Q98: '<think>\nOkay, let's tackle this question step by ...' → 5
  Q99: '<think>\nOkay, let's tackle this question. The use...' → 4
  Q100: '<think>\nOkay, let's tackle this question. The use...' → 3

Done — answered 100 / 100 questions


---
# Section 8: Export CSV สำหรับ Submission

In [ ]:
with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    # เปิดไฟล์ไว้สำหรับเขียนผลลัพธ์

    writer = csv.writer(f)

    # เขียนหัวตาราง
    writer.writerow(["id", "answer"])

    # วนทุกคำถาม แล้วเขียนคำตอบลงไฟล์
    for q in questions:
        qid = q["id"]

        # ถ้ามีคำตอบใน hybrid_preds ก็ใช้ค่านั้น
        # ถ้าไม่มี ให้ใช้ค่า 1 แทน (กันพลาด)
        writer.writerow([qid, hybrid_preds.get(qid, 1)])

# แจ้งว่าเขียนไฟล์เสร็จแล้ว
print(f"submission.csv written ({len(questions)} rows)")

submission.csv written (100 rows)
